# Extraction des données pour la fusion

La méthode de réccupération des données est assez simple, il existe plusieur api permettant cela. Dans cette première partie du pipeline concentré sur la réccupération des des données one ne les travailles pas et on s'occupe juste de mettre chaque type de données dnas un seul csv.

## Imports & variables

In [2]:
import os
import requests
import pandas as pd
import gzip
import zipfile
import time
# from io import BytesIO

# Dossier de sortie pour les données finales
output_folder = "../data/extraction/"

# Dossier temporaire pour les fichiers téléchargés / compressés
tmp_folder = "../data/extraction/tmp/"

# Préfixes utilisés pour nommer les différents jeux de données
meteo_name = "meteo"
nappe_name = "nappes"
etp_name = "etp"
impermeabilite_name = "impermeabilite"

## Réccupération automatique des données Météo (ancienne)

Le problème de météo france est le fait qu'a l'heure actuel on doit téléchargé les mois 1 par 1.

In [ ]:
# output_folder = "../data/extraction/tmp/meteo"
# os.makedirs(output_folder, exist_ok=True)

# start_year = 1990
# end_year = 2025

# base_url = "https://donneespubliques.meteofrance.fr/donnees_libres/Txt/Climat/climat.{date}.csv.gz"

# for year in range(start_year, end_year + 1):
#     for month in range(1, 13):
#         date_str = f"{year}{month:02d}"
#         url = base_url.format(date=date_str)
#         local_file = os.path.join(output_folder, f"climat.{date_str}.csv.gz")

#         if os.path.exists(local_file):
#             print(f"[SKIP] {local_file} existe déjà")
#             continue

#         try:
#             r = requests.get(url, timeout=10)
#             if r.status_code == 200:
#                 with open(local_file, "wb") as f:
#                     f.write(r.content)
#                 print(f"[OK] Téléchargé {date_str}")
#             else:
#                 print(f"[MANQUANT] {date_str} (HTTP {r.status_code})")
#         except Exception as e:
#             print(f"[ERREUR] {date_str} : {e}")

# print("Téléchargement terminé !")

In [ ]:
# input_folder = "../data/extraction/tmp/meteo"
# output_file = "../data/extraction/meteo.csv"

# all_files = [f for f in os.listdir(input_folder) if f.endswith(".csv.gz")]
# all_files.sort()
# dfs = []

# for file in all_files:
#     file_path = os.path.join(input_folder, file)
    
#     with gzip.open(file_path, 'rt', encoding='utf-8') as f:
#         df = pd.read_csv(f, sep=";")
#         dfs.append(df)

# meteo_complet = pd.concat(dfs, ignore_index=True)

# meteo_complet.to_csv(output_file, sep=";", index=False)

# print(f"[OK] {len(all_files)} fichiers fusionnés dans {output_file}")

## Réccupération automatique des données Météos

Ici j'utilise les données de : https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-mensuelles

Afin de pouvoir réccupérer les information pour communiquer avec l'api : https://www.data.gouv.fr/api/1/datasets/{dataset_id}

Pour les données météos on sait donc que le dataset est : 6569b3d7d193b4daf2b43edc

In [ ]:
# ID du dataset météo sur data.gouv
dataset_id = "6569b3d7d193b4daf2b43edc"

# Dossier temporaire pour les fichiers météo
os.makedirs(f"{tmp_folder}/{meteo_name}", exist_ok=True)

# Récupération des métadonnées du dataset
url = f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}"
r = requests.get(url)
data = r.json()
resources = data["resources"]

urls_45 = []

# Filtrer uniquement les fichiers du département 45/Loiret
for res in resources:
    name = (res["title"] + res["url"]).lower()
    if "_45" in name:
        urls_45.append(res["url"])
        print("Trouvé :", res["title"])

if not urls_45:
    print("Aucun fichier pour le département 45")
    exit()

print(f"{len(urls_45)} fichiers trouvés")

all_dfs = []

# Téléchargement et lecture de chaque fichier compressé
for i, file_url in enumerate(urls_45):
    print(f"Téléchargement {i+1}/{len(urls_45)}")

    local_file = f"{tmp_folder}/{meteo_name}/tmp_{i}.csv.gz"
    r = requests.get(file_url)

    # Sauvegarde locale du fichier
    with open(local_file, "wb") as f:
        f.write(r.content)

    # Lecture du CSV compressé
    df = pd.read_csv(local_file, sep=";", compression="gzip")
    all_dfs.append(df)

# Fusion de tous les fichiers en un seul DataFrame
df_final = pd.concat(all_dfs, ignore_index=True)

# Export du fichier final
output_path = f"{output_folder}/{meteo_name}.csv"
df_final.to_csv(output_path, sep=";", index=False)

print("Fichier final créé :", output_path)
print("Nombre total de lignes :", len(df_final))

## Réccupération automatique des fichiers du niveau de l'eau des nappes

Ici j'utilise les données de l'api : https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations

In [ ]:
departement = "45"

# Dossier de sortie pour les fichiers nappes
os.makedirs(f"{output_folder}/{nappe_name}", exist_ok=True)

url_stations = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"

codes = []
page = 1

# Récupération paginée des stations du département
while True:
    params = {
        "code_departement": departement,
        "size": 200,
        "page": page
    }
    
    r = requests.get(url_stations, params=params)
    data = r.json()["data"]
    
    if not data:
        break

    # Stockage des codes des stations et des coordonnées
    codes.extend([(s["code_bss"],s["x"],s["y"]) for s in data])
    print(f"Page {page} : {len(data)} stations")
    page += 1

# Suppression des doublons
codes = list(set(codes))
print(f"Total stations trouvées : {len(codes)}")

url_chroniques = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"

# Téléchargement des chroniques pour chaque station
for i, (code, lat, lon) in enumerate(codes):
    params = {
        "code_bss": code,
        "size": 20000
    }
    
    try:
        r = requests.get(url_chroniques, params=params)
        data = r.json()["data"]
        
        if not data:
            print(f"[{i+1}/{len(codes)}] {code} : aucune donnée")
            continue
        
        df = pd.DataFrame(data)
        df["lat"] = lat
        df["lon"] = lon
        
        # Nettoyage du nom pour le fichier
        safe_code = code.replace("/", "_")
        file_path = os.path.join(f"{output_folder}/{nappe_name}", f"{nappe_name}_{safe_code}.csv")
        
        df.to_csv(file_path, sep=";", index=False)
        print(f"[{i+1}/{len(codes)}] {code} : {len(df)} lignes")
        
        # Pause pour éviter de surcharger l'API
        time.sleep(0.2)
        
    except Exception as e:
        print(f"[{i+1}/{len(codes)}] Erreur {code} : {e}")

print("Téléchargement terminé")

## Réccupération automatique des fichiers d'ETP

Ici j'utilise les données de : https://www.data.gouv.fr/datasets/etp-fao-hargreaves

Afin de pouvoir réccupérer les information pour communiquer avec l'api : https://www.data.gouv.fr/api/1/datasets/{dataset_id}

Pour les données de l'évapotranspiration on sait donc que le dataset est : 667eae35510cd549fc7722c1

/!\ il faut penser à aller tellecharger par soit meme le fichier : https://donneespubliques.meteofrance.fr/client/document/metadonnees_swi_276.csv
de le renomé maille.csv, de le mettre dans le dossier data et de retirer les 3 4 premières lignes qui donne des indications.

In [ ]:
# ID du dataset météo sur data.gouv
dataset_id = "667eae35510cd549fc7722c1"

# Dossier temporaire pour les fichiers ETP
os.makedirs(f"{tmp_folder}/{etp_name}", exist_ok=True)

# Récupération des ressources du dataset
resp = requests.get(f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}/")
data = resp.json()

# Téléchargement des fichiers CSV / CSV compressés
for resource in data.get("resources", []):
    url = resource.get("url")
    fmt = resource.get("format", "").lower()
    if fmt in ["csv", "csv.gz"]:
        filename = os.path.join(f"{tmp_folder}/{etp_name}", url.split("/")[-1])
        print(f"[DOWNLOAD] {url}")
        r = requests.get(url)
        with open(filename, "wb") as f:
            f.write(r.content)

all_dfs = []

print("Lecture des fichiers...")

# Fichier de correspondance des mailles
maille_file = "../data/maille.csv" 
maille_df = pd.read_csv(maille_file, sep=";")

# coo très très approximatif du Loiret
LAT_MIN, LAT_MAX = 47.416069, 48.392186
LON_MIN, LON_MAX = 1.474425, 3.221622

# Filtrage des mailles situées dans le Loiret
maille_loiret = maille_df[
    (maille_df["lat_dg"] >= LAT_MIN) & (maille_df["lat_dg"] <= LAT_MAX) &
    (maille_df["lon_dg"] >= LON_MIN) & (maille_df["lon_dg"] <= LON_MAX)
]

maille_loiret_xy = maille_loiret[["lambx", "lamby", "lat_dg", "lon_dg",]]

print("Nombre de mailles Loiret :", len(maille_loiret))
mailles_loiret = set(maille_loiret["num_maille"])

# Lecture et filtrage des fichiers téléchargés
for file in os.listdir(f"{tmp_folder}/{etp_name}"):
    path = os.path.join(f"{tmp_folder}/{etp_name}", file)
    print(path)
    
    if file.endswith(".csv"):
        df = pd.read_csv(path, sep=";") 
    elif file.endswith(".csv.gz"):
        # Gestion des différents formats compressés, ici j'ai pas compris les fichier sont en .gz mais sont encodé en zip
        try:
            with zipfile.ZipFile(path, 'r') as z:
                for name in z.namelist():
                    if name.endswith(".csv"):
                        with z.open(name) as f:
                            df = pd.read_csv(f, sep=";")

        # Pour je ne sais qu'elle raison le dernier fichier et bien encodé en gz
        except :
            with gzip.open(path, 'rt') as f:
                df = pd.read_csv(f, sep=";")
        
    
    # Filtrage des mailles du Loiret
    if "NUM_MAILLE" in df.columns:
        df = df[df["NUM_MAILLE"].isin(mailles_loiret)]
    elif "LAMBX" in df.columns and "LAMBY" in df.columns:
        df = df.merge(
            maille_loiret_xy,
            left_on=["LAMBX", "LAMBY"],
            right_on=["lambx", "lamby"],
            how="inner"
        )
    
    all_dfs.append(df)

# Fusion finale
if all_dfs:
    etp_all = pd.concat(all_dfs, ignore_index=True)
    etp_all = etp_all[["DATE","ETP_Q_H0175","lat_dg","lon_dg"]]
    
    if "DATE" in etp_all.columns:
        etp_all = etp_all.sort_values("DATE")
    
    etp_all.to_csv(f"{output_folder}/{etp_name}.csv", sep=";", index=False)
    
    print("Nombre de lignes :", len(etp_all))
else:
    print("Aucun fichier trouvé")

## Réccupération automatique des fichiers d'imperméabilité des sols

Ici j'utilise les données de : https://www.data.gouv.fr/datasets/etp-fao-hargreaves

Afin de pouvoir réccupérer les information pour communiquer avec l'api : https://www.data.gouv.fr/api/1/datasets/{dataset_id}

Pour les données de l'évapotranspiration on sait donc que le dataset est : 697b4f4ceea77fb452ba9d6d

/!\ il faut penser à aller tellecharger par soit meme le fichier : https://www.data.gouv.fr/datasets/donnees-sur-les-communes-de-france-metropolitaine
de le renomé communes.csv, de le mettre dans le dossier data.

In [ ]:
dataset_id = "697b4f4ceea77fb452ba9d6d"
communes_file = "../data/communes.csv"

# Création des dossiers
os.makedirs(f"{tmp_folder}/{impermeabilite_name}", exist_ok=True)
os.makedirs("../output", exist_ok=True)

# Récupération des données du dataset
resp = requests.get(f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}/")
data = resp.json()

print("Téléchargement des ressources...")
for resource in data.get("resources", []):
    url = resource.get("url")
    fmt = resource.get("format","").lower()
    
    # On ne télécharge que les fichiers CSV
    if fmt == "csv":
        filename = os.path.join(
            f"{tmp_folder}/{impermeabilite_name}",
            url.split("/")[-1]
        )
        print(f"[DOWNLOAD] {url}")
        r = requests.get(url)
        with open(filename, "wb") as f:
            f.write(r.content)

# Lecture et fusion des fichiers du dataset (données communales)
all_dfs = []
print("Lecture des fichiers d'imperméabilisation...")

for file in os.listdir(f"{tmp_folder}/{impermeabilite_name}"):
    if file.endswith("commune.csv"):
        path = os.path.join(f"{tmp_folder}/{impermeabilite_name}", file)
        # On force le type string pour le code commune pour éviter de perdre les zéros initiaux
        df = pd.read_csv(path, sep=",", dtype={'commune_code': str})
        all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)

# Lecture du référentiel des communes (lat/lon)
print("Lecture du fichier communes.csv...")
# On force le type string pour Code Insee
communes = pd.read_csv(communes_file,sep=",", dtype={'Code Insee': str})

communes = communes[communes['département'] == 'Loiret']

# Sélection des colonnes utiles du référentiel
communes = communes[[
    "Code Insee",
    "latitude",
    "longitude"
]]


#  Fusion sur commune_code = Code Insee
# On s'assure que les codes font 5 caractères (ex: 01001)
df_all['commune_code'] = df_all['commune_code'].str.zfill(5)
communes['Code Insee'] = communes['Code Insee'].str.zfill(5)

df_merged = df_all.merge(
    communes,
    left_on="commune_code",
    right_on="Code Insee",
    how="inner"
)

# Transformation au format long (un record par millésime)
print("Transformation des données...")
records = []

for _, row in df_merged.iterrows():
    records.append({
        "lat": row["latitude"],
        "lon": row["longitude"],
        "time": row["millesimes_1"],
        "surface_imp": row["surface_imper_1"],
        "surface": row["commune_surface"]
    })
    records.append({
        "lat": row["latitude"],
        "lon": row["longitude"],
        "time": row["millesimes_2"],
        "surface_imp": row["surface_imper_2"],
        "surface": row["commune_surface"]
    })

df_final = pd.DataFrame(records)

# Tri par temps et localisation
df_final = df_final.sort_values(by=["time", "lat", "lon"])

# Exportation
df_final.to_csv(
    f"{output_folder}/{impermeabilite_name}.csv",
    sep=";",
    index=False
)

print(f"\nFichier créé : {output_folder}/{impermeabilite_name}.csv")
print(f"Nombre de lignes finales : {len(df_final)}")
print(df_final.head())

Téléchargement des ressources...
[DOWNLOAD] https://static.data.gouv.fr/resources/impermeabilisation-des-collectivites-de-france/20260129-121620/imper-commune.csv
[DOWNLOAD] https://static.data.gouv.fr/resources/impermeabilisation-des-collectivites-de-france/20260129-121603/imper-epci.csv
[DOWNLOAD] https://static.data.gouv.fr/resources/impermeabilisation-des-collectivites-de-france/20260129-121603/imper-scot.csv
[DOWNLOAD] https://static.data.gouv.fr/resources/impermeabilisation-des-collectivites-de-france/20260129-121559/imper-departement.csv
[DOWNLOAD] https://static.data.gouv.fr/resources/impermeabilisation-des-collectivites-de-france/20260129-121555/imper-region.csv
Lecture des fichiers d'imperméabilisation...
Lecture du fichier communes.csv...
Transformation des données...

Fichier créé : ../data/extraction//impermeabilite.csv
Nombre de lignes finales : 698
           lat       lon    time   surface_imp       surface
510  47.510953  2.692078  2020.0  4.098469e+05  2.767014e+07
54